In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [7]:
# 1. LOAD DATA 
df = pd.read_excel(r"C:\Users\burug\OneDrive\Desktop\projects\Sales performance\salesforcourse-4fe2kehu.xlsx")

In [6]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [8]:
print("=" * 55)
print("STEP 1: RAW DATA OVERVIEW")
print("=" * 55)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst 3 Rows:")
print(df.head(3).to_string())

STEP 1: RAW DATA OVERVIEW
Rows    : 34,867
Columns : 16

Column Names:
['index', 'Date', 'Year', 'Month', 'Customer Age', 'Customer Gender', 'Country', 'State', 'Product Category', 'Sub Category', 'Quantity', 'Unit Cost', 'Unit Price', 'Cost', 'Revenue', 'Column1']

First 3 Rows:
   index       Date    Year     Month  Customer Age Customer Gender        Country       State Product Category     Sub Category  Quantity  Unit Cost  Unit Price  Cost  Revenue  Column1
0      0 2016-02-19  2016.0  February          29.0               F  United States  Washington      Accessories  Tires and Tubes       1.0      80.00       109.0  80.0    109.0      NaN
1      1 2016-02-20  2016.0  February          29.0               F  United States  Washington         Clothing           Gloves       2.0      24.50        28.5  49.0     57.0      NaN
2      2 2016-02-27  2016.0  February          29.0               F  United States  Washington      Accessories  Tires and Tubes       3.0       3.67         5.0

In [9]:
# 2. DROP USELESS COLUMNS ──
# 'Column1' has 32,000+ nulls out of 34,867 rows — useless
# 'index'   is just row numbers — not needed
print("\n" + "=" * 55)
print("STEP 2: DROP USELESS COLUMNS")
print("=" * 55)

cols_to_drop = ['index', 'Column1']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Dropped: {cols_to_drop}")
print(f"Remaining columns: {df.columns.tolist()}")


STEP 2: DROP USELESS COLUMNS
Dropped: ['index', 'Column1']
Remaining columns: ['Date', 'Year', 'Month', 'Customer Age', 'Customer Gender', 'Country', 'State', 'Product Category', 'Sub Category', 'Quantity', 'Unit Cost', 'Unit Price', 'Cost', 'Revenue']


In [10]:
#3 Remove nulls
print("\n" + "=" * 55)
print("STEP 3: NULL VALUES")
print("=" * 55)
print("Missing values per column:")
print(df.isnull().sum())


STEP 3: NULL VALUES
Missing values per column:
Date                1
Year                1
Month               1
Customer Age        1
Customer Gender     1
Country             1
State               1
Product Category    1
Sub Category        1
Quantity            1
Unit Cost           1
Unit Price          1
Cost                1
Revenue             0
dtype: int64


In [13]:
# Drop the one row that has nulls across all columns
before = df.shape[0]
df = df.dropna()
after = df.shape[0]
print(f"\nRows dropped: {before - after}")
print(f"Clean rows remaining: {after:,}")


Rows dropped: 0
Clean rows remaining: 34,866


In [14]:
#4 Check duplicates 
print("\n" + "=" * 55)
print("STEP 4: DUPLICATES")
print("=" * 55)
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"Removed. New shape: {df.shape}")
else:
    print("No duplicates found.")


STEP 4: DUPLICATES
Duplicate rows: 1
Removed. New shape: (34865, 14)


In [15]:
# 5. STANDARDISE COLUMN NAMES ─
print("\n" + "=" * 55)
print("STEP 5: STANDARDISE COLUMN NAMES")
print("=" * 55)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)
print("Cleaned column names:")
print(df.columns.tolist())


STEP 5: STANDARDISE COLUMN NAMES
Cleaned column names:
['date', 'year', 'month', 'customer_age', 'customer_gender', 'country', 'state', 'product_category', 'sub_category', 'quantity', 'unit_cost', 'unit_price', 'cost', 'revenue']


In [16]:
#  6. FIX DATA TYPES 
print("\n" + "=" * 55)
print("STEP 6: FIX DATA TYPES")
print("=" * 55)
 
# Date column — ensure it's datetime
df['date'] = pd.to_datetime(df['date'])
print(f"'date' column type: {df['date'].dtype}")
 
# Quantity should be integer
df['quantity'] = df['quantity'].astype(int)
print(f"'quantity' column type: {df['quantity'].dtype}")
 
# Year should be integer
df['year'] = df['year'].astype(int)
print(f"'year' column type: {df['year'].dtype}")
 
# Customer Age should be integer
df['customer_age'] = df['customer_age'].astype(int)
print(f"'customer_age' column type: {df['customer_age'].dtype}")
 
 


STEP 6: FIX DATA TYPES
'date' column type: datetime64[us]
'quantity' column type: int64
'year' column type: int64
'customer_age' column type: int64


In [17]:
# 7. STANDARDISE TEXT COLUMNS 
print("\n" + "=" * 55)
print("STEP 7: STANDARDISE TEXT VALUES")
print("=" * 55)
text_cols = df.select_dtypes(include='object').columns.tolist()
for col in text_cols:
    df[col] = df[col].str.strip().str.title()
 
print("Unique values in key columns:")
print(f"  Country          : {df['country'].unique()}")
print(f"  Product Category : {df['product_category'].unique()}")
print(f"  Customer Gender  : {df['customer_gender'].unique()}")
print(f"  Year             : {sorted(df['year'].unique())}")


STEP 7: STANDARDISE TEXT VALUES
Unique values in key columns:
  Country          : <StringArray>
['United States', 'France', 'United Kingdom', 'Germany']
Length: 4, dtype: str
  Product Category : <StringArray>
['Accessories', 'Clothing', 'Bikes']
Length: 3, dtype: str
  Customer Gender  : <StringArray>
['F', 'M']
Length: 2, dtype: str
  Year             : [np.int64(2015), np.int64(2016)]


In [18]:
#8 Key Metrics
print("\n" + "=" * 55)
print("STEP 8: DERIVE KEY METRICS")
print("=" * 55)
 
# Profit = Revenue - Cost
df['profit'] = df['revenue'] - df['cost']
print(f"  Profit column created. Sample: {df['profit'].head(3).tolist()}")
 
# Profit Margin % = (Profit / Revenue) * 100
df['profit_margin_pct'] = ((df['profit'] / df['revenue']) * 100).round(2)
print(f"  Profit Margin % created. Sample: {df['profit_margin_pct'].head(3).tolist()}")
 
# Age Group — group customers into brackets
def age_group(age):
    if age <= 25:
        return '18-25'
    elif age <= 35:
        return '26-35'
    elif age <= 45:
        return '36-45'
    elif age <= 55:
        return '46-55'
    else:
        return '55+'
 
df['age_group'] = df['customer_age'].apply(age_group)
print(f"  Age Group column created.")
print(f"  Age groups: {sorted(df['age_group'].unique())}")


STEP 8: DERIVE KEY METRICS
  Profit column created. Sample: [29.0, 8.0, 4.0]
  Profit Margin % created. Sample: [26.61, 14.04, 26.67]
  Age Group column created.
  Age groups: ['18-25', '26-35', '36-45', '46-55', '55+']


In [19]:
#9 Summary Statistics
print("\n" + "=" * 55)
print("STEP 9: BUSINESS SUMMARY")
print("=" * 55)
print(f"  Total Revenue : ${df['revenue'].sum():>15,.2f}")
print(f"  Total Cost    : ${df['cost'].sum():>15,.2f}")
print(f"  Total Profit  : ${df['profit'].sum():>15,.2f}")
print(f"  Avg Margin %  : {df['profit_margin_pct'].mean():>15.2f}%")
print(f"  Total Orders  : {df.shape[0]:>15,}")
print(f"  Avg Order Val : ${df['revenue'].mean():>15,.2f}")
 
print("\nRevenue by Country:")
print(df.groupby('country')['revenue'].sum().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))
 
print("\nRevenue by Product Category:")
print(df.groupby('product_category')['revenue'].sum().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))
 
print("\nRevenue by Year:")
print(df.groupby('year')['revenue'].sum().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))


STEP 9: BUSINESS SUMMARY
  Total Revenue : $  22,344,548.00
  Total Cost    : $  20,082,954.00
  Total Profit  : $   2,261,594.00
  Avg Margin %  :           13.41%
  Total Orders  :          34,865
  Avg Order Val : $         640.89

Revenue by Country:
country
United States     $10,377,742.00
United Kingdom     $4,276,220.00
Germany            $4,244,482.00
France             $3,446,104.00
Name: revenue, dtype: str

Revenue by Product Category:
product_category
Bikes          $11,486,355.00
Accessories     $7,420,636.00
Clothing        $3,437,557.00
Name: revenue, dtype: str

Revenue by Year:
year
2016    $12,396,805.00
2015     $9,947,743.00
Name: revenue, dtype: str


In [20]:
#  10. EXPORT CLEANED DATA 
output_path = r"C:\Users\burug\OneDrive\Desktop\projects\Sales performance\sales_cleaned.csv"
df.to_csv(output_path, index=False)
 
print("\n" + "=" * 55)
print("STEP 10: EXPORT COMPLETE")
print("=" * 55)
print(f"Saved to: {output_path}")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"New columns added: profit, profit_margin_pct, age_group")
print("\nData cleaning complete. Ready for SQL analysis.")


STEP 10: EXPORT COMPLETE
Saved to: C:\Users\burug\OneDrive\Desktop\projects\Sales performance\sales_cleaned.csv
Final shape: 34,865 rows x 17 columns
New columns added: profit, profit_margin_pct, age_group

Data cleaning complete. Ready for SQL analysis.
